# Install packages

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# !pip install openai groq python-dotenv

# Import libraries

In [ ]:
import os
from google.colab import files
import pandas as pd
import time
import json
import re  # For parsing verdict
from groq import Groq  
from dotenv import load_dotenv

# Select LLM_as_a_Judge

In [ ]:
'''
LLM-as-a-judge models selected from Groq API provider:
['openai/gpt-oss-20b', 'llama-3.3-70b-versatile', 'moonshotai/kimi-k2-instruct', 'qwen/qwen3-32b', 'llama-3.1-8b-instant']

Duplicate this notebook to run per LLM-as-a-judge
Adjust judge_model accordingly
'''
judge_model = 'openai/gpt-oss-20b'

# Set up paths

In [ ]:
def extract_model_name(model_string):
  match = re.match(r'[^/-]+', model_string)
  if match:
    return match.group(0)
  else:
    return model_string

In [ ]:
# If judge_model is 'llama-3.1-8b-instant', the judge_name will be 'tinyllama'
judge_name = extract_model_name(judge_model)
file_name = 'pair_llama_deepseek'
# Adjust file path to match the target environment
file_path = f'drive/MyDrive/ViJudge/data/datasets_before_judgement/{file_name}.csv'
judge_prompts_path = 'drive/MyDrive/ViJudge/prompts/judge_prompts.jsonl'
output_path = f'drive/MyDrive/ViJudge/data/position_bias/{judge_name}_position_bias_for_{file_name}.csv'

# Upload file .env to Colab

In [ ]:
uploaded_file = files.upload()

# Get API Key

In [ ]:
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

if not groq_api_key:
    raise ValueError("GROQ_API_KEY not found")

# Connect with Groq

In [ ]:
groq_client = Groq(
    api_key = groq_api_key,
)

# Preprocessing

In [ ]:
# List of questions selected for position bias evaluation
question_list = [16, 5, 11, 8, 24, 20, 29, 28, 59, 42, 56, 53, 75, 79, 61, 62, 83, 85, 89, 97]

In [ ]:
df = pd.read_csv(file_path)
df.head()

In [ ]:
def filter_and_swap_dataframe(df, question_list):
    """
    Filter dataframe by selected questions and create two variants:
    - df1: preserve order, keep rows with question_id in question_list
    - df2: same as df1 with model A and B swapped

    Args:
    - df: input DataFrame with question_id formatted as "abcdefg"
    - question_list: list of target question IDs (middle 2-digit substring, e.g., "bc")

    Returns:
    - df1: original filtered DataFrame
    - df2: filtered DataFrame with model A/B swapped
    """

    # Extract the middle two digits from question_id
    def extract_middle_digits(question_id):
        # Extract the middle two digits from question_id formatted as 'abcdefg'
        question_id_str = str(question_id)
        if len(question_id_str) >= 3:
            # Extract characters at positions 1 and 2 (zero-based indexing)
            return question_id_str[1:3]
        return None

    # Create a temporary column for comparison
    df_copy = df.copy()
    df_copy['middle_digits'] = df_copy['question_id'].apply(extract_middle_digits)

    # Convert question_list to string with 2 digits for comparison
    # zfill(2) will add leading zeros if the number has only 1 digit
    # Example: 0 -> "00", 5 -> "05", 12 -> "12"
    question_list_str = [str(q).zfill(2) for q in question_list]

    # Filter dataframe 1: preserve original order
    df1 = df_copy[df_copy['middle_digits'].isin(question_list_str)].copy()
    df1 = df1.drop(columns=['middle_digits'])

    # Create dataframe 2: swap model A and B
    df2 = df1.copy()

    # Swap column names: A -> temp, B -> A, temp -> B
    columns_to_swap = {
        'model_a': 'model_b',
        'model_b': 'model_a',
        'response_turn1_a': 'response_turn1_b',
        'response_turn1_b': 'response_turn1_a',
        'response_turn2_a': 'response_turn2_b',
        'response_turn2_b': 'response_turn2_a',
    }

    # Perform swap by exchanging values
    for col_a, col_b in [('model_a', 'model_b'),
                          ('response_turn1_a', 'response_turn1_b'),
                          ('response_turn2_a', 'response_turn2_b')]:
        if col_a in df2.columns and col_b in df2.columns:
            df2[col_a], df2[col_b] = df2[col_b].copy(), df2[col_a].copy()

    return df1, df2


In [ ]:
df1, df2 = filter_and_swap_dataframe(df, question_list)
display(df1.head())
display(df2.head())

# Load prompts

In [ ]:
# Load Judge Prompts
judge_prompts = {}
with open(judge_prompts_path, 'r') as f:
    for line in f:
        prompt_data = json.loads(line)
        judge_prompts[prompt_data['name']] = prompt_data
print(f"Loaded prompts: {list(judge_prompts.keys())}")

# Generate judment

In [ ]:
# Function to Generate Judgement
def generate_judgement(client, row, judge_name='openai/gpt-oss-20b'):
    # Extract values, handle NaN
    question_1 = row['content_turn1'] if pd.notna(row['content_turn1']) else ''
    question_2 = row['content_turn2'] if pd.notna(row['content_turn2']) else ''
    ref_1 = row['reference_turn1'] if pd.notna(row['reference_turn1']) else ''
    ref_2 = row['reference_turn2'] if pd.notna(row['reference_turn2']) else ''
    answer_1_a = row['response_turn1_a'] if pd.notna(row['response_turn1_a']) else ''
    answer_2_a = row['response_turn2_a'] if pd.notna(row['response_turn2_a']) else ''
    answer_1_b = row['response_turn1_b'] if pd.notna(row['response_turn1_b']) else ''
    answer_2_b = row['response_turn2_b'] if pd.notna(row['response_turn2_b']) else ''
    model_a = row['model_a'] if pd.notna(row['model_a']) else ''
    model_b = row['model_b'] if pd.notna(row['model_b']) else ''
    
    # Is multi-turn if has question_2
    multi_turn = bool(question_2)

    # Has reference if either ref_1 or ref_2 is not empty
    has_ref = bool(ref_1 or ref_2)

    # Is pair if has response b, else single
    judge_type = 'pair' if answer_1_b or answer_2_b else 'single'

    # Define prompt name
    if multi_turn:
        prompt_name = judge_type + '_multi_turn'
    else:
        prompt_name = judge_type + '_1_turn'

    if has_ref:
        prompt_name = prompt_name + '_reference'

    if prompt_name not in judge_prompts:
        raise ValueError(f'Prompt {prompt_name} not found')

    prompt_data = judge_prompts[prompt_name]
    system_prompt = prompt_data['system_prompt']
    template = prompt_data['prompt_template']

    # Format user_prompt
    if multi_turn:
        if judge_type == 'pair':
            user_prompt = template.format(
                question_1=question_1, question_2=question_2,
                ref_answer_1=ref_1, ref_answer_2=ref_2,
                answer_a_1=answer_1_a, answer_a_2=answer_2_a,
                answer_b_1=answer_1_b, answer_b_2=answer_2_b
            )
        else:  # single multi-turn
            user_prompt = template.format(
                question_1=question_1, question_2=question_2,
                ref_answer_1=ref_1, ref_answer_2=ref_2,
                answer_1=answer_1_a, answer_2=answer_2_a
            )
    else:
        # Single-turn:
        if '{question_2}' in template:
            raise ValueError('Template not compatible with single-turn')
        if judge_type == 'pair':
            user_prompt = template.format(
                question=question_1,
                ref_answer_1=ref_1,
                answer_a=answer_1_a,
                answer_b=answer_1_b
            )
        else:
            user_prompt = template.format(
                question=question_1,
                ref_answer_1=ref_1,
                answer=answer_1_a
            )

    # Call API
    try:
        response = client.chat.completions.create(
            model=judge_name,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0.0,
            max_tokens=2048
        )
        judgement = response.choices[0].message.content.strip()
    except Exception as e:
        error_message = str(e)
        if "Request too large" in error_message and "413" in error_message:
            print(f"Request too large for q_id below -> verdict = 'unknown'")
            judgement = 'unknown'
        else:
            raise e # Re-raise exceptions other than 'Request too large'

    verdict_match = re.search(r'\[\[(A|B|C|\d+)\]\]', judgement)
    verdict = verdict_match.group(1) if verdict_match else "Unknown"

    conversation_a = [
        {"content": question_1, "role": "user"},
        {"content": answer_1_a, "role": "assistant"},
        {"content": question_2, "role": "user"},
        {"content": answer_2_a, "role": "assistant"},
    ]
    conversation_b = [
        {"content": question_1, "role": "user"},
        {"content": answer_1_b, "role": "assistant"},
        {"content": question_2, "role": "user"},
        {"content": answer_2_b, "role": "assistant"},
    ]

    new_row = {
        'question_id': row['question_id'],
        'category': row['category'],
        'model_a': model_a,
        'model_b': model_b,
        'conversation_a': conversation_a,
        'conversation_b': conversation_b,
        'judge_name': judge_name,
        'judge_type': judge_type,
        'judgement_explanation': judgement,
        'verdict': verdict
    }

    return new_row

In [ ]:
def process_position_bias_judgements(df, output_path, question_list, client, judge_model, n=2):
    """
    Evaluate position bias by running both original and swapped configurations

    Args:
    - df: input DataFrame loaded from CSV
    - output_path: path to save results CSV
    - question_list: list of target question IDs (integers)
    - client: API client for judge model calls
    - judge_model: judge model name
    - n: number of evaluation runs per configuration (default=2)

    Output:
    - Writes CSV with columns:
    question_id, category, model_a, model_b, conversation_a, conversation_b,
    judge_name, judge_type, judgement_explanation, verdict,
    position_config (original/swapped), attempt (1,2,...,n)
    """

    # Create df1 (original) and df2 (swapped)
    df1, df2 = filter_and_swap_dataframe(df, question_list)

    # Load successfully processed question_ids
    existing_records = set()
    if os.path.exists(output_path):
        existing_df = pd.read_csv(output_path)
        # Create key in format: "question_id-position_config-attempt"
        for _, row in existing_df.iterrows():
            key = f"{row['question_id']}-{row['position_config']}-{row['attempt']}"
            existing_records.add(key)
        print(f"Loaded {len(existing_records)} existing records from {output_path}")

    # List of configurations to run
    configs = []
    for attempt in range(1, n + 1):
        # Add original configuration
        for _, row in df1.iterrows():
            configs.append({
                'row': row,
                'position_config': 'original',
                'attempt': attempt,
                'question_id': row['question_id']
            })
        # Add swapped configuration
        for _, row in df2.iterrows():
            configs.append({
                'row': row,
                'position_config': 'swapped',
                'attempt': attempt,
                'question_id': row['question_id']
            })

    print(f"Total configurations to process: {len(configs)}")

    # Process each configuration
    success_count = 0
    skip_count = 0
    error_count = 0

    for config in configs:
        row = config['row']
        position_config = config['position_config']
        attempt = config['attempt']
        q_id = config['question_id']

        # Create key to check if already processed
        record_key = f"{q_id}-{position_config}-{attempt}"

        if record_key in existing_records:
            print(f"⏭️  Skipping {record_key} (already processed)")
            skip_count += 1
            continue

        try:
            print(f"🔄 Processing {record_key}...")

            # Call generate_judgement()
            judgement = generate_judgement(client, row, judge_model)

            # Add info about position_config and attempt
            judgement['position_config'] = position_config
            judgement['attempt'] = attempt

            # Save to CSV (append)
            temp_df = pd.DataFrame([judgement])
            if os.path.exists(output_path):
                temp_df.to_csv(output_path, mode='a', header=False, index=False)
            else:
                temp_df.to_csv(output_path, mode='w', header=True, index=False)

            # Add to existing_records to avoid reprocessing on restart
            existing_records.add(record_key)

            success_count += 1
            print(f"✅ Success: {record_key} | Verdict: {judgement['verdict']}")

            # Rate limit
            time.sleep(10)

        except Exception as e:
            error_count += 1
            print(f"❌ Error on {record_key}: {e}")
            continue

    print(f"\n{'='*50}")
    print(f"📊 Summary:")
    print(f"   ✅ Success: {success_count}")
    print(f"   ⏭️  Skipped: {skip_count}")
    print(f"   ❌ Errors: {error_count}")
    print(f"   📁 Results saved to: {output_path}")
    print(f"{'='*50}")

In [ ]:
# Run
process_position_bias_judgements(
    df=df,
    output_path=output_path,
    question_list=question_list,
    client=groq_client,
    judge_model=judge_model,
    n=4
)

# Check

In [ ]:
result = pd.read_csv(output_path)
print(f"\nTotal results: {len(result)}")
print(f"\nPosition config distribution:")
print(result['position_config'].value_counts())
print(f"\nAttempt distribution:")
print(result['attempt'].value_counts())
result.head(len(result))